In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from openbox import space as sp
import pandas as pd
import xgboost
import numpy as np
import matplotlib.pyplot as plt
from gpytorch.priors import GammaPrior
from edbo.bro import BO

%matplotlib inline

In [3]:
df_yield = pd.read_csv('../data/flow2.tsv', sep='\t')

# 对所有负数取正
df_yield['T_drawH'] = df_yield['T_drawH'].apply(lambda x: -x)
df_yield['T_methyl'] = df_yield['T_methyl'].apply(lambda x: -x)

features = df_yield.drop(['yield'], axis=1)
y = df_yield['yield']
X = pd.DataFrame(features ,columns=features.columns)

In [4]:
X.head()

,m_B,t_RT1,T_drawH,m_C,t_RT2,T_methyl
0,2.66,8.45,39.0,3.15,121.2,30
1,2.65,4.23,39.0,3.15,121.2,30
2,2.66,2.07,39.0,3.15,121.2,30
3,2.13,4.81,39.0,3.15,121.2,30
4,3.13,3.70,39.0,3.15,121.2,30


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [45]:
# train an XGBoost model
model = xgboost.XGBRegressor(
    learning_rate = 0.1,
        subsample = 0.8,
        colsample_bynode = 0.2,
        reg_lambda = 1e-5,).fit(X_train, y_train)

In [47]:
# predict the known best
X_preds = model.predict(X)
print(X_preds)

[54.896633 74.46128  72.746765 65.020164 71.625595 24.57056  40.152378
 39.696175 46.271503 48.77172  45.38713  49.365788 61.42682  58.01092
 74.65097  78.53971  82.945915 82.33664  86.07178  83.32855  80.89075
 81.670296 59.340797 75.372925 82.29472  84.084816 80.56371  79.75521
 79.14052  81.05601 ]


In [48]:
print(f'r2 score for train set is: {r2_score(y_train, model.predict(X_train))}')
print(f'r2 score for test set is: {r2_score(y_test, model.predict(X_test))}')

r2 score for train set is: 0.9999263464932823
r2 score for test set is: 0.7962927744664091


In [49]:
import joblib
from pathlib import Path

# save model
model_path = Path('./methylation/model')
model_path.mkdir(exist_ok=True)

joblib.dump(model, model_path / 'xgboost.bin')

['methylation/model/xgboost.bin']